# Weights & Biases (W&B): Experiment Tracking & Collaboration

## What is W&B?
Weights & Biases is a MLOps platform for tracking experiments, visualizing metrics, comparing models, managing datasets, and collaborating on ML projects.

## Core Features
| Feature | Description |
|---------|------------|
| **Runs** | Single training execution with all logged data |
| **Projects** | Group of related runs |
| **Sweeps** | Hyperparameter optimization |
| **Artifacts** | Versioned datasets, models, and files |
| **Reports** | Interactive dashboards and documentation |
| **Tables** | Interactive data visualization |

## Setup
```bash
pip install wandb
wandb login  # enter API key from wandb.ai/authorize
```

In [1]:
# pip install wandb torch torchvision
import wandb
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print(f"W&B version: {wandb.__version__}")

W&B version: 0.27.2


## Basic W&B Logging

In [2]:
# Load data
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Convert to tensors
X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.FloatTensor(y_train)
X_test_t  = torch.FloatTensor(X_test)
y_test_t  = torch.FloatTensor(y_test)

# Define model
class BinaryClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x).squeeze()

# Initialize W&B run
config = {
    "learning_rate": 0.001,
    "epochs": 50,
    "hidden_dim": 64,
    "dropout": 0.3,
    "optimizer": "adam",
    "dataset": "breast_cancer"
}

# Comment out wandb.init for offline testing or set mode="offline"
# run = wandb.init(project="breast_cancer_classification", config=config, mode="offline")
# Simulated training loop (works without W&B connection):

model = BinaryClassifier(X_train.shape[1], config["hidden_dim"], config["dropout"])
optimizer = optim.Adam(model.parameters(), lr=config["learning_rate"])
criterion = nn.BCELoss()

for epoch in range(config["epochs"]):
    model.train()
    optimizer.zero_grad()
    pred = model(X_train_t)
    loss = criterion(pred, y_train_t)
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        val_pred = model(X_test_t)
        val_loss = criterion(val_pred, y_test_t).item()
        accuracy = ((val_pred > 0.5).float() == y_test_t).float().mean().item()

    # wandb.log({"train_loss": loss.item(), "val_loss": val_loss, "accuracy": accuracy, "epoch": epoch})
    if epoch % 10 == 0:
        print(f"Epoch {epoch}: train_loss={loss.item():.4f}, val_loss={val_loss:.4f}, acc={accuracy:.4f}")

# wandb.finish()
print("Training complete")

Epoch 0: train_loss=0.6943, val_loss=0.6850, acc=0.5965
Epoch 10: train_loss=0.5959, val_loss=0.5741, acc=0.8421
Epoch 20: train_loss=0.4764, val_loss=0.4459, acc=0.9386
Epoch 30: train_loss=0.3377, val_loss=0.3066, acc=0.9649
Epoch 40: train_loss=0.2292, val_loss=0.1962, acc=0.9649
Training complete


## wandb.watch() Gradient & Parameter Tracking

In [3]:
# wandb.watch() hooks into the model to log gradients and parameters

# with wandb.init(project="breast_cancer_classification", mode="offline") as run:
#     model = BinaryClassifier(X_train.shape[1], 64)
#     optimizer = optim.Adam(model.parameters(), lr=0.001)
#     criterion = nn.BCELoss()
#
#     # Watch model logs gradients every 10 steps
#     wandb.watch(model, criterion, log="all", log_freq=10)
#
#     for epoch in range(50):
#         pred = model(X_train_t)
#         loss = criterion(pred, y_train_t)
#         optimizer.zero_grad()
#         loss.backward()
#         optimizer.step()
#         wandb.log({"loss": loss.item()})

print("wandb.watch() tracks gradients and weights great for debugging training")

wandb.watch() tracks gradients and weights great for debugging training


## W&B Artifacts Versioned Datasets & Models

In [4]:
import os
import pandas as pd

# Save dataset as artifact
# with wandb.init(project="breast_cancer_classification", job_type="data-prep", mode="offline") as run:
#     # Create artifact
#     artifact = wandb.Artifact(
#         name="breast_cancer_dataset",
#         type="dataset",
#         description="Breast cancer dataset from sklearn",
#         metadata={"source": "sklearn", "samples": len(X), "features": X.shape[1]}
#     )
#
#     # Add files to artifact
#     df = pd.DataFrame(X, columns=data.feature_names)
#     df['target'] = y
#     df.to_csv('/tmp/breast_cancer.csv', index=False)
#     artifact.add_file('/tmp/breast_cancer.csv')
#
#     # Log artifact
#     run.log_artifact(artifact)
#
# # Use artifact in training run
# with wandb.init(project="breast_cancer_classification", job_type="train", mode="offline") as run:
#     # Download artifact
#     artifact = run.use_artifact('breast_cancer_dataset:latest')
#     data_dir = artifact.download()
#     print(f"Data downloaded to: {data_dir}")

print("Artifacts enable data versioning track what data was used for each model")

Artifacts enable data versioning track what data was used for each model


## W&B Sweeps Hyperparameter Optimization

Sweeps automate hyperparameter search with three strategies:
- **Grid search**: Try all combinations
- **Random search**: Sample randomly
- **Bayesian**: Use past results to guide next trial

In [5]:
# Sweep configuration
sweep_config = {
    "method": "bayes",   # 'grid', 'random', 'bayes'
    "metric": {
        "name": "val_accuracy",
        "goal": "maximize"
    },
    "early_terminate": {
        "type": "hyperband",
        "min_iter": 5
    },
    "parameters": {
        "learning_rate": {
            "distribution": "log_uniform_values",
            "min": 1e-5,
            "max": 1e-1
        },
        "hidden_dim": {
            "values": [32, 64, 128, 256]
        },
        "dropout": {
            "distribution": "uniform",
            "min": 0.1,
            "max": 0.5
        },
        "epochs": {
            "value": 50
        }
    }
}

# Training function for sweep
def train_sweep():
    # with wandb.init() as run:
    #     config = wandb.config
    #     model = BinaryClassifier(X_train.shape[1], config.hidden_dim, config.dropout)
    #     optimizer = optim.Adam(model.parameters(), lr=config.learning_rate)
    #     criterion = nn.BCELoss()
    #     for epoch in range(config.epochs):
    #         model.train()
    #         pred = model(X_train_t)
    #         loss = criterion(pred, y_train_t)
    #         optimizer.zero_grad(); loss.backward(); optimizer.step()
    #         model.eval()
    #         with torch.no_grad():
    #             val_pred = model(X_test_t)
    #             acc = ((val_pred > 0.5).float() == y_test_t).float().mean().item()
    #         wandb.log({"val_accuracy": acc, "train_loss": loss.item()})
    pass

# Initialize and run sweep:
# sweep_id = wandb.sweep(sweep_config, project="breast_cancer_classification")
# wandb.agent(sweep_id, function=train_sweep, count=20)  # run 20 trials

print("Sweep config defined. In production: wandb.sweep() + wandb.agent()")

Sweep config defined. In production: wandb.sweep() + wandb.agent()


## W&B Tables Dataset Visualization

In [6]:
# W&B Tables for visualizing predictions
# with wandb.init(project="breast_cancer_classification", mode="offline") as run:
#     model.eval()
#     with torch.no_grad():
#         preds = model(X_test_t).numpy()
#         labels = y_test
#
#     # Create prediction table
#     columns = list(data.feature_names) + ["true_label", "predicted_prob", "predicted_label"]
#     table_data = []
#     for i in range(len(X_test)):
#         row = list(X_test[i]) + [int(labels[i]), float(preds[i]), int(preds[i] > 0.5)]
#         table_data.append(row)
#
#     table = wandb.Table(columns=columns, data=table_data)
#     run.log({"predictions": table})
#
#     # Confusion matrix
#     wandb.log({"confusion_matrix": wandb.plot.confusion_matrix(
#         probs=None,
#         y_true=labels.tolist(),
#         preds=[int(p > 0.5) for p in preds],
#         class_names=["Malignant", "Benign"]
#     )})

print("Tables and built-in plots make W&B great for debugging model predictions")

Tables and built-in plots make W&B great for debugging model predictions


## W&B for LLMs Prompt Tracking

In [7]:
# W&B Weave for LLM tracking
# pip install weave

# import weave
# weave.init('llm-experiment')
#
# @weave.op()
# def generate_response(prompt: str, model: str = "gpt-4o") -> str:
#     from openai import OpenAI
#     client = OpenAI()
#     response = client.chat.completions.create(
#         model=model,
#         messages=[{"role": "user", "content": prompt}]
#     )
#     return response.choices[0].message.content
#
# result = generate_response("Explain transformers in one paragraph")
# # All calls automatically tracked in Weave

# Traditional W&B LLM logging
# with wandb.init(project="llm-experiments", mode="offline") as run:
#     prompt_table = wandb.Table(columns=["prompt", "response", "model", "tokens", "latency_ms"])
#     prompt_table.add_data("Explain AI", "AI is...", "gpt-4o", 150, 320)
#     run.log({"prompt_log": prompt_table})

print("W&B Weave: https://weave-docs.wandb.ai/ purpose-built LLM observability")

W&B Weave: https://weave-docs.wandb.ai/ purpose-built LLM observability


## Additional Learning Resources

### Official Documentation
- [W&B Docs](https://docs.wandb.ai/) Complete reference
- [W&B Quickstart](https://docs.wandb.ai/quickstart)
- [W&B Sweeps](https://docs.wandb.ai/guides/sweeps)
- [W&B Artifacts](https://docs.wandb.ai/guides/artifacts)
- [W&B Weave](https://weave-docs.wandb.ai/) LLM tracking

### Tutorials
- [W&B Courses](https://www.wandb.courses/) Free courses
- [Effective MLOps: Model Development](https://www.wandb.courses/courses/effective-mlops-model-development)
- [MLOps Zoomcamp](https://github.com/DataTalksClub/mlops-zoomcamp) W&B module included

### Papers
- [Weights & Biases: Machine Learning Experiment Tracking](https://arxiv.org/abs/2006.04683)